<a href="https://colab.research.google.com/github/Gokul-k11/data-science/blob/Q-%26-A-System/question_and_answer_system.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install -q pypdf sentence-transformers faiss-cpu transformers torch

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 349.5/349.5 kB 19.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.5/18.5 MB 71.3 MB/s eta 0:00:00


In [28]:
from google.colab import files

uploaded = files.upload()
pdf_path = list(uploaded.keys())[0]
print(f"Uploaded: {pdf_path}")

Saving lehs101.pdf to lehs101.pdf
Saving lehs102.pdf to lehs102.pdf
Uploaded: lehs101.pdf


In [29]:
from pypdf import PdfReader

reader = PdfReader(pdf_path)
pages_text = []

for i, page in enumerate(reader.pages):
    text = page.extract_text() or ""
    if text.strip():
        pages_text.append((i + 1, text))

print(f"Extracted text from {len(pages_text)} pages")

Extracted text from 27 pages


In [30]:
def chunk_page(page_num, text, chunk_size=150, overlap=30):
    words = text.split()
    chunks = []
    i = 0
    while i < len(words):
        chunk = " ".join(words[i:i + chunk_size])
        chunks.append({"page": page_num, "text": chunk})
        i += chunk_size - overlap
    return chunks

all_chunks = []
for page_num, text in pages_text:
    all_chunks.extend(chunk_page(page_num, text))

print(f"Created {len(all_chunks)} chunks across {len(pages_text)} pages")

Created 84 chunks across 27 pages


In [31]:
from sentence_transformers import SentenceTransformer
import faiss
import numpy as np

embedder = SentenceTransformer("all-MiniLM-L6-v2")
chunk_texts = [c["text"] for c in all_chunks]
embeddings = embedder.encode(chunk_texts, show_progress_bar=True)

dimension = embeddings.shape[1]
index = faiss.IndexFlatL2(dimension)
index.add(np.array(embeddings).astype("float32"))

print(f"Indexed {index.ntotal} chunks")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Batches:   0%|          | 0/3 [00:00<?, ?it/s]

Indexed 84 chunks


In [ ]:
!pip install -q --upgrade --force-reinstall transformers

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.5/40.5 kB 3.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.3/57.3 kB 5.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.6/11.6 MB 136.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 780.4/780.4 kB 58.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.7/16.7 MB 109.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 100.2/100.2 kB 10.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 807.9/807.9 kB 55.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 801.8/801.8 kB 63.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 516.0/516.0 kB 49.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 44.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 80.2/80.2 kB 8.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 122.7/122.7 kB 13.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [32]:
import torch
from transformers import AutoTokenizer, AutoModelForQuestionAnswering

model_name = "distilbert-base-cased-distilled-squad"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForQuestionAnswering.from_pretrained(model_name)
model.eval()

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

print("Model loaded successfully")

Loading weights:   0%|          | 0/102 [00:00<?, ?it/s]

Model loaded successfully


In [33]:
def answer_from_context(question, context):
    inputs = tokenizer(
        question,
        context,
        return_tensors="pt",
        truncation=True,
        max_length=384,
        return_offsets_mapping=True
    )

    offset_mapping = inputs.pop("offset_mapping")[0]
    sequence_ids = inputs.sequence_ids(0)

    inputs = {k: v.to(device) for k, v in inputs.items()}

    with torch.no_grad():
        outputs = model(**inputs)

    start_logits = outputs.start_logits[0]
    end_logits = outputs.end_logits[0]

    # Mask out tokens that aren't part of the context (question + special tokens)
    for i, seq_id in enumerate(sequence_ids):
        if seq_id != 1:  # 1 = context tokens, 0 = question tokens, None = special tokens
            start_logits[i] = -1e9
            end_logits[i] = -1e9

    start_idx = torch.argmax(start_logits).item()
    end_idx = torch.argmax(end_logits).item()

    if end_idx < start_idx:
        end_idx = start_idx

    start_char = offset_mapping[start_idx][0].item()
    end_char = offset_mapping[end_idx][1].item()

    exact_answer = context[start_char:end_char]

    score = float(
        torch.softmax(start_logits, dim=0)[start_idx] *
        torch.softmax(end_logits, dim=0)[end_idx]
    )

    return exact_answer.strip(), score

In [34]:
def retrieve_top_chunks(query, k=3):
    query_vec = embedder.encode([query]).astype("float32")
    distances, indices = index.search(query_vec, k)
    return [all_chunks[i] for i in indices[0]]

def answer_question(question, k=3):
    candidates = retrieve_top_chunks(question, k)

    best_answer = None
    best_score = -1
    best_page = None

    for c in candidates:
        answer, score = answer_from_context(question, c["text"])
        if answer.strip() and score > best_score:
            best_score = score
            best_answer = answer
            best_page = c["page"]

    if best_score < 0.05 or not best_answer:
        return "This isn't clearly covered in the provided document.", None, None

    return best_answer, best_page, best_score

In [38]:
import re

while True:
    question = input("\nAsk a question about your PDF (or type 'exit'): ")
    if question.lower().strip() == "exit":
        break

    candidates = retrieve_top_chunks(question, k=3)

    best_answer = None
    best_score = -1
    best_context = None

    for c in candidates:
        answer, score = answer_from_context(question, c["text"])
        if answer and score > best_score:
            best_score = score
            best_answer = answer
            best_context = c["text"]

    if not best_answer or best_score < 0.01:
        print("\nThis isn't clearly covered in the provided document.")
    else:
        print(f"\nAnswer: {best_answer}")
        print("\nContent:")

        # Split into sentences and print each on its own line
        sentences = re.split(r'(?<=[.!?])\s+', best_context.strip())
        for line in sentences:
            if line.strip():
                print(line.strip())


Ask a question about your PDF (or type 'exit'): Domestic architecture

Answer: 7 3.2

Content:
7 3.2 Domestic architecture The Lower Town at Mohenjodaro provides examples of residential buildings.
Many were centred on a courtyard, with rooms on all sides.
The courtyard was probably the centre of activities such as cooking and weaving, particularly during hot and dry weather.
What is also interesting is an apparent concern for privacy: there are no windows in the walls along the ground level.
Besides, the main entrance does not give a direct view of the interior or the courtyard.
Every house had its own bathroom paved with bricks, with drains connected through the wall to the street drains.
Some houses have remains of staircases to reach a second storey or the roof.
Many houses had wells, often in a room that could be reached from the outside and perhaps used by passers-by.
Scholars have estimated that the total number of wells in Mohenjodaro was about 700.
The most

Ask a question abo

In [ ]:
!pip install -q gradio

In [36]:
import csv
import os
import re

HISTORY_FILE = "qa_history.csv"

# Create history file with headers if it doesn't exist
if not os.path.exists(HISTORY_FILE):
    with open(HISTORY_FILE, "w", newline="", encoding="utf-8") as f:
        writer = csv.writer(f)
        writer.writerow(["Question", "Answer", "Confidence", "Page", "Content"])

def highlight_answer(context, answer):
    if not answer:
        return context
    pattern = re.escape(answer)
    highlighted = re.sub(
        f"({pattern})",
        r"**\1**",
        context,
        count=1,
        flags=re.IGNORECASE
    )
    return highlighted

def save_to_history(question, answer, score, page, content):
    with open(HISTORY_FILE, "a", newline="", encoding="utf-8") as f:
        writer = csv.writer(f)
        writer.writerow([question, answer, f"{score:.4f}", page, content])

def get_answer(question, k=3):
    if not question.strip():
        return "Please enter a question.", ""

    candidates = retrieve_top_chunks(question, k)

    best_answer = None
    best_score = -1
    best_context = None
    best_page = None

    for c in candidates:
        answer, score = answer_from_context(question, c["text"])
        if answer and score > best_score:
            best_score = score
            best_answer = answer
            best_context = c["text"]
            best_page = c["page"]

    if not best_answer or best_score < 0.01:
        result_text = "This isn't clearly covered in the provided document."
        save_to_history(question, "NOT FOUND", 0, "-", "-")
        return result_text, ""

    highlighted = highlight_answer(best_context, best_answer)
    save_to_history(question, best_answer, best_score, best_page, best_context)

    answer_display = f"**Answer:** {best_answer}\n\n**Confidence:** {best_score:.2f}  |  **Page:** {best_page}"
    content_display = highlighted

    return answer_display, content_display

In [37]:
import gradio as gr

with gr.Blocks(title="PDF Question Answering System") as demo:
    gr.Markdown("# 📚 Textbook Question Answering System")
    gr.Markdown("Ask any question about your uploaded PDF. The answer is extracted directly from the document.")

    with gr.Row():
        question_input = gr.Textbox(
            label="Your Question",
            placeholder="e.g. What is the main topic of chapter 1?",
            lines=2
        )

    with gr.Row():
        k_slider = gr.Slider(1, 10, value=3, step=1, label="Number of chunks to search (k)")

    ask_btn = gr.Button("Ask", variant="primary")

    answer_output = gr.Markdown(label="Answer")
    content_output = gr.Markdown(label="Source Content (answer highlighted)")

    ask_btn.click(
        fn=get_answer,
        inputs=[question_input, k_slider],
        outputs=[answer_output, content_output]
    )

    question_input.submit(
        fn=get_answer,
        inputs=[question_input, k_slider],
        outputs=[answer_output, content_output]
    )

demo.launch(share=True, debug=True)

Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://35bd6340c0c5e6a7ee.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


Keyboard interruption in main thread... closing server.
Killing tunnel 127.0.0.1:7860 <> https://35bd6340c0c5e6a7ee.gradio.live
